# WE_pairwise_qwen_v2 — Qwen chấm từng cặp, ghép vào ensemble 0.73

## Ý tưởng tham khảo và phần triển khai
**Ý tưởng tham khảo** (do người dùng cung cấp): chấm độ tương đồng từng cặp từ bằng LLM, ghép điểm thành **ma trận tương đồng**, rồi dựng **một vector tĩnh cho mỗi từ** sao cho cosine phản ánh điểm.

**Bản v2 khác bản v1 ở ba điểm** (mục 1–3 đã thống nhất):
1. **Nền mạnh hơn:** `S_base` là cosine của **ensemble 0.73** (thay cho Numberbatch + WordNet ≈ 0.6). Công thức: `S = z(S_ens) + α·z(S_qwen)`.
2. **Đo riêng sức mạnh của Qwen:** xuất `WE_Qwen_only.zip` (không có ensemble) để bạn nộp và biết Qwen tự đạt bao nhiêu, kèm `WE_Base_Ensemble_655d.zip` để kiểm tra định dạng 655 chiều (kỳ vọng ≈ 0.73).
3. **Điểm Qwen tốt hơn:** chấm **cả hai thứ tự** (A,B) và (B,A); hỗ trợ **mô hình lớn** (7B, nạp 4-bit); **trung bình nhiều prompt** (`sim`, `subst`, `kind`); tuỳ chọn **few-shot** bằng cặp RG-65 kèm điểm người chấm. Có bảng so sánh các cấu hình trên dev trước khi chạy đủ.

Mỗi *thành phần* = (prompt, thứ tự) được chấm cho cả 214.185 cặp và **cache riêng**. Nhờ vậy bạn có thể thêm prompt/thứ tự ở phiên sau mà không chấm lại phần đã có, và nếu hết giờ phiên thì chạy tiếp.

> **Không dùng WordSim-353 hoặc SimLex-999 ở bất kỳ bước nào** (không nhãn, không để chọn tham số, không trong prompt). Ví dụ few-shot lấy từ RG-65 và **loại** các cặp có cả hai từ nằm trong 655 từ. SimVerb không dùng.

## Bạn cần chuẩn bị gì
| # | Cần có | Ghi chú |
|---|---|---|
| 1 | `starting-kit.py` | upload thành Kaggle Dataset (hoặc vào thư mục làm việc) |
| 2 | **File ensemble 0.73** `WE_ensemble6_B2_F2_L2_strong_C_E.zip` (hoặc `WE.bin` giải nén) | upload thành Kaggle Dataset; notebook tự tìm, hoặc đặt `ENSEMBLE_PATH` |
| 3 | Kaggle: **GPU** (T4 ×2 hoặc P100) và **Internet = On** | Internet cần cho tải Qwen, dev set; hoặc dùng *Add Input → Models → Qwen2.5* rồi đặt `MODEL_NAME` là đường dẫn |
| 4 | Thời gian | mỗi thành phần chấm 214.185 cặp; **chưa đo tốc độ**, smoke-test sẽ in ước tính giờ thật |

## Trạng thái kiểm thử (đọc trước khi chạy)
- **Đã kiểm tra** (ngoài Kaggle, với bộ chấm Qwen *giả*): đọc starting-kit, đọc ensemble thật, dựng vector 655 chiều và kiểm tra thứ hạng, cache theo thành phần, chạy tiếp, giới hạn giờ, xuất `Qwen_only`/`Combined`/`Base`, bảng so sánh dev.
- **Chưa kiểm tra:** chạy **Qwen thật** (nạp 4-bit/bitsandbytes, tương thích `transformers`, VRAM, tốc độ, độ chính xác). Smoke-test thiết kế để kiểm tra đúng phần này trước khi tốn giờ.
- Điểm dev chỉ là kiểm tra cục bộ (RG-65/MC-30 nhỏ, sai số ≈ ±0.03, đã bão hoà), **không dự đoán điểm hệ thống chấm**. Notebook **không chọn α bằng dev** (từ dev không tính được nền ensemble vì các từ dev không nằm trong 655 từ); bạn chọn α bằng cách nộp.

## Cách chạy trên Kaggle
1. **Import Notebook** (File → Import Notebook). Thêm hai Dataset ở mục 1 và 2 (Add Input).
2. **Settings:** Accelerator = GPU; Internet = On.
3. **Lần 1 — `SMOKE_TEST = True` (mặc định):** chạy tới cuối. Notebook xuất `WE_Base_Ensemble_655d.zip` (**nộp thử ngay** để chắc hệ thống chấm nhận 655 chiều và cho ≈ 0.73), in **bảng so sánh cấu hình trên dev**, đo tốc độ và in **ước tính giờ cho từng thành phần**.
4. Chỉnh cấu hình theo bảng dev và ngân sách giờ (`PROMPT_SET`, `BOTH_ORDERS`, `USE_FEWSHOT`, mô hình).
5. **Lần 2 — `SMOKE_TEST = False`:** *Save Version → Save & Run All (Commit)* để chạy nền. `STOP_AFTER_HOURS` (mặc định 10,5) dừng êm trước giới hạn phiên của Kaggle (kiểm tra giới hạn hiện hành trên trang Kaggle; mình không xác nhận số liệu) và lưu cache. Dùng `COMPONENTS_PER_RUN` để chỉ chạy một số thành phần mỗi phiên.
6. **Tiếp tục ở phiên sau:** thêm **output của bản chạy trước** làm Input, đặt `INPUT_CACHE_DIR` = thư mục `cache` trong đó; cache chỉ được dùng nếu **mô hình, prompt, few-shot và danh sách cặp** khớp hoàn toàn.
7. Tải các ZIP ở tab **Output**. Mỗi ZIP chỉ chứa `WE.bin` ở thư mục gốc.

## Gợi ý nộp và cách đọc điểm
1. `WE_Base_Ensemble_655d.zip` — kỳ vọng ≈ 0.73. Thấp hơn rõ rệt nghĩa là định dạng/số chiều có vấn đề, dừng lại kiểm tra.
2. `WE_Qwen_only.zip` — sức mạnh riêng của Qwen. **≥ 0.70:** hướng đáng đi tiếp. **Dưới ~0.60:** khó giúp ensemble.
3. `WE_Combined_alpha*.zip` — so với 0.73. Nếu α lớn nhất tốt nhất, Qwen mạnh hơn ensemble: thử α lớn hơn bằng `ALPHA_LIST`.

In [1]:
%pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.1 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ===================== CẤU HÌNH (chỉnh ở đây) =====================
import os, re, sys, json, ast, math, time, zipfile, hashlib, shutil, itertools, random, urllib.request
import numpy as np
from scipy.stats import spearmanr
T_START = time.time()

IS_KAGGLE = os.path.exists('/kaggle/working')
WORK = '/kaggle/working' if IS_KAGGLE else os.getcwd()

# ---- Đầu vào ----
STARTING_KIT  = None        # None = tự tìm 'starting-kit.py'
ENSEMBLE_PATH = None        # zip hoặc WE.bin của ensemble 0.73; None = tự tìm trong /kaggle/input

# ---- Qwen ----
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'   # hoặc 'Qwen/Qwen2.5-3B-Instruct' (nhanh hơn), hoặc đường dẫn mô hình trong /kaggle/input/...
LOAD_4BIT  = True           # 7B trên 1 GPU 16 GB cần 4-bit (bitsandbytes). Đặt False cho 3B.
SINGLE_GPU = True           # True = dùng GPU 0; False = chia mô hình qua các GPU (vd. 7B fp16 trên 2 GPU)
DTYPE      = 'float16'      # 'float16' (T4/P100) | 'bfloat16' (A100/H100) | 'float32' (nếu gặp NaN)
BATCH_SIZE = 32             # giảm nếu hết VRAM

# ---- Cấu hình chấm điểm (mục 3) ----
PROMPT_SET  = ['sim']       # chọn từ: 'sim' (giống nghĩa), 'subst' (thay thế được cho nhau), 'kind' (cùng loại). Thêm sau: cache giữ nguyên.
BOTH_ORDERS = True          # chấm cả (A,B) và (B,A)
USE_FEWSHOT = False         # thêm N_SHOTS ví dụ RG-65 (kèm điểm người chấm) vào prompt; tăng độ dài prompt nên chậm hơn
N_SHOTS     = 8

# ---- Chế độ chạy ----
SMOKE_TEST   = False         # True = chỉ chấm N_SMOKE cặp để kiểm tra code/VRAM/tốc độ. ĐỔI SANG False ĐỂ CHẠY ĐỦ.
N_SMOKE      = 512
CHECKPOINT_EVERY   = 10_000
COMPONENTS_PER_RUN = None   # None = chạy hết; hoặc số thành phần (prompt,thứ tự) tối đa mỗi phiên
STOP_AFTER_HOURS   = 10.5   # dừng êm (lưu cache) khi notebook đã chạy chừng này giờ; None = không giới hạn
SESSION_HOURS_BUDGET = 9.0  # chỉ dùng để cảnh báo trong bảng ước tính giờ
ABORT_IF_QWEN_DEV_BELOW = 0.50
CACHE_DIR       = f'{WORK}/cache'
INPUT_CACHE_DIR = None      # vd. '/kaggle/input/<output-cu>/cache' để tiếp tục

# ---- Xuất ----
ALPHA_LIST = [0.5, 1, 2]    # trọng số Qwen so với ensemble; mỗi giá trị > 0 là một file nộp
OUT_DIM    = None           # None = giữ đủ 655 chiều (thứ hạng cosine đúng tuyệt đối). Số nguyên < 655 = cắt chiều (xấp xỉ, xem độ trung thực)
OUT_DIR    = f'{WORK}/out'

os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
print('Kaggle:', IS_KAGGLE, '| thư mục làm việc:', WORK, '| mô hình:', MODEL_NAME, '| 4-bit:', LOAD_4BIT)
print('SMOKE_TEST =', SMOKE_TEST, '| PROMPT_SET =', PROMPT_SET, '| BOTH_ORDERS =', BOTH_ORDERS, '| USE_FEWSHOT =', USE_FEWSHOT)
if SMOKE_TEST: print('>>> SMOKE_TEST: sẽ KHÔNG xuất bản Qwen. Sau khi xem bảng dev và ước tính giờ, đặt SMOKE_TEST = False và chạy lại.')

Kaggle: True | thư mục làm việc: /kaggle/working | mô hình: Qwen/Qwen2.5-7B-Instruct | 4-bit: True
SMOKE_TEST = False | PROMPT_SET = ['sim'] | BOTH_ORDERS = True | USE_FEWSHOT = False


## 1. Từ vựng và danh sách cặp
Đọc `Words` từ `starting-kit.py` bằng `ast` (không chạy file), giữ nguyên thứ tự; kiểm tra 655 từ duy nhất → 214.185 cặp `(i<j)`.

In [3]:
def find_starting_kit():
    cands = [STARTING_KIT] if STARTING_KIT else []
    cands += ['starting-kit.py', 'starting_kit.py', f'{WORK}/starting-kit.py']
    if IS_KAGGLE:
        for root, _, files in os.walk('/kaggle/input'):
            cands += [os.path.join(root, f) for f in files if f.lower().startswith('starting') and f.endswith('.py')]
    for c in cands:
        if c and os.path.exists(c): return c
    raise FileNotFoundError('Không thấy starting-kit.py: upload (Dataset hoặc thư mục làm việc) hoặc đặt STARTING_KIT.')

def read_words(path):
    tree = ast.parse(open(path, encoding='utf8').read())
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(getattr(t, 'id', None) == 'Words' for t in node.targets):
            return list(ast.literal_eval(node.value))
    raise ValueError('Không thấy `Words = [...]` trong starting-kit')

kit = find_starting_kit(); Words = read_words(kit); n = len(Words); WSET = set(Words)
assert n == 655 and len(WSET) == 655, f'Cần 655 từ duy nhất, đang có {n} ({len(WSET)} duy nhất)'
I, J = np.triu_indices(n, 1); NP = len(I); assert NP == 214_185
PAIR_WORDS = [(Words[i], Words[j]) for i, j in zip(I, J)]
PAIR_WORDS_BA = [(b, a) for a, b in PAIR_WORDS]
print(f'starting-kit: {kit}\n{n} từ duy nhất, {NP:,} cặp. 5 từ đầu: {Words[:5]} ... cuối: {Words[-5:]}')

starting-kit: /kaggle/input/datasets/tttung279999/nlp-lab1/starting-kit.py
655 từ duy nhất, 214,185 cặp. 5 từ đầu: ['ability', 'absence', 'abundance', 'accident', 'actor'] ... cuối: ['world', 'wrist', 'yard', 'young', 'zone']


## 2. Nền: cosine của ensemble 0.73
Đọc `WE.bin` của ensemble (zip hoặc đã giải nén), **đối chiếu 655 từ và sắp xếp lại đúng thứ tự `Words`**, chuẩn hoá L2 và tính cosine cho 214.185 cặp → `S_ens`.

In [4]:
from gensim.models import KeyedVectors
def find_ensemble():
    if ENSEMBLE_PATH: return ENSEMBLE_PATH
    found = []
    for base in (['/kaggle/input'] if IS_KAGGLE else []) + [WORK, '.']:
        for root, _, files in os.walk(base):
            for f in files:
                p = os.path.join(root, f)
                if (f.lower().endswith('.zip') or f == 'WE.bin') and 'ensemble' in p.lower(): found.append(p)
    pref = [p for p in found if 'ensemble6' in p.lower()] or found
    if not pref: raise FileNotFoundError('Không thấy file ensemble: upload WE_ensemble6_B2_F2_L2_strong_C_E.zip thành Dataset hoặc đặt ENSEMBLE_PATH.')
    if len(pref) > 1: print('⚠ nhiều ứng viên, dùng cái đầu:', pref)
    return pref[0]

def load_embedding(path):
    tmp = f'{WORK}/_emb_tmp'; shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    if path.lower().endswith('.zip'):
        with zipfile.ZipFile(path) as z: assert 'WE.bin' in z.namelist(), z.namelist(); z.extract('WE.bin', tmp)
        bin_path = f'{tmp}/WE.bin'
    else: bin_path = path
    kv = KeyedVectors.load_word2vec_format(bin_path, binary=True); shutil.rmtree(tmp, ignore_errors=True)
    assert set(kv.index_to_key) == WSET, 'file ensemble không đúng bộ 655 từ'
    V = kv.vectors[[kv.key_to_index[w] for w in Words]].astype(np.float64)
    return V / np.linalg.norm(V, axis=1, keepdims=True)

ens_path = find_ensemble(); XE = load_embedding(ens_path)
COS_ENS = (XE[I] * XE[J]).sum(1)
print(f'ensemble: {ens_path} | {XE.shape[1]} chiều | cosine 214.185 cặp: min {COS_ENS.min():.3f}, TB {COS_ENS.mean():.3f}, max {COS_ENS.max():.3f}')

ensemble: /kaggle/input/datasets/tttung279999/we-ensemble6-b2-f2-l2-strong-c-e/WE.bin | 300 chiều | cosine 214.185 cặp: min -0.552, TB 0.024, max 0.996


## 3. Dựng vector từ điểm cặp, xuất ZIP

1. Chuẩn hoá z-score điểm 214.185 cặp, đặt vào ma trận đối xứng `S` (**đường chéo = 0**).
2. `λ_min` = trị riêng nhỏ nhất; cộng **cùng một hằng số** `c = −λ_min + 1e-3` vào đường chéo ⇒ `G = S + c·I` bán xác định dương.
3. `G = V Λ Vᵀ`, vector `X = V √Λ`, **giữ toàn bộ chiều** (655). Mọi vector có cùng chuẩn `√c` nên `cosine(i,j) = S_ij / c`: **thứ hạng cosine = thứ hạng điểm** (kiểm tra bằng Spearman sau khi đọc lại `WE.bin`).
4. `OUT_DIM` (tuỳ chọn): lấy `OUT_DIM` trị riêng lớn nhất thay vì đủ 655; **không còn đúng tuyệt đối**, độ trung thực được in ra.

Cosine đầu ra nằm trong khoảng nhỏ quanh 0 (≈ ±0,1–0,2); vô hại với đánh giá theo thứ hạng. Số chiều 655 = số từ (hạng tối đa của ma trận); **kiểm tra hệ thống chấm nhận 655 chiều** bằng cách nộp `WE_Base_Ensemble_655d.zip` trước.

In [5]:
zscore = lambda v, p=None: ((v - v.mean()) / v.std(), (v.mean(), v.std())) if p is None else (v - p[0]) / p[1]

def vectors_from_scores(score_pairs, out_dim=None):
    S = np.zeros((n, n)); S[I, J] = score_pairs; S[J, I] = score_pairs
    lam = np.linalg.eigvalsh(S); c = max(0.0, -lam.min()) + 1e-3
    w, V = np.linalg.eigh(S + c * np.eye(n)); assert w.min() > 0, 'G chưa xác định dương'
    if out_dim is not None and out_dim < n: w, V = w[-out_dim:], V[:, -out_dim:]
    return V * np.sqrt(w), dict(lambda_min=float(lam.min()), c=float(c), dims=int(len(w)), eig_min=float(w.min()), eig_max=float(w.max()))

def validate_zip(zip_path, score_pairs):
    with zipfile.ZipFile(zip_path) as z:
        assert z.namelist() == ['WE.bin'], z.namelist()
        d = f'{OUT_DIR}/_chk'; os.makedirs(d, exist_ok=True); z.extract('WE.bin', d)
    kv = KeyedVectors.load_word2vec_format(f'{d}/WE.bin', binary=True); shutil.rmtree(d)
    assert kv.index_to_key == Words, 'sai thứ tự/bộ từ'
    V = kv.vectors; assert np.isfinite(V).all() and (np.linalg.norm(V, axis=1) > 0).all()
    Vn = V / np.linalg.norm(V, axis=1, keepdims=True); cosv = (Vn[I] * Vn[J]).sum(1)
    return dict(words=len(kv.index_to_key), dims_file=int(kv.vector_size), spearman_cos_vs_score=float(spearmanr(cosv, score_pairs)[0]), cos_min=float(cosv.min()), cos_max=float(cosv.max()))

def export_zip(score_pairs, zip_path):
    X, info = vectors_from_scores(score_pairs, OUT_DIM)
    tmp = f'{OUT_DIR}/_tmp_{os.path.basename(zip_path)}'; os.makedirs(tmp, exist_ok=True)
    kv = KeyedVectors(vector_size=X.shape[1]); kv.add_vectors(Words, X.astype(np.float32)); kv.save_word2vec_format(f'{tmp}/WE.bin', binary=True)
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z: z.write(f'{tmp}/WE.bin', 'WE.bin')      # WE.bin ở thư mục gốc
    shutil.rmtree(tmp); info.update(validate_zip(zip_path, score_pairs)); return info

ZE, ZE_P = zscore(COS_ENS)
BASE_ZIP = f'{OUT_DIR}/WE_Base_Ensemble_655d.zip'
info_base = export_zip(ZE, BASE_ZIP); print('Đã xuất:', BASE_ZIP); print(json.dumps(info_base, indent=1))
if OUT_DIM is None: assert info_base['spearman_cos_vs_score'] > 0.9999, 'thứ hạng cosine không khớp điểm đầu vào'
STATUS = {'base_zip': BASE_ZIP, 'base_exported': True}
print('\n>>> Nên nộp thử file này trước: kỳ vọng ≈ 0.73 (nếu thấp hơn rõ rệt, kiểm tra định dạng 655 chiều).')

Đã xuất: /kaggle/working/out/WE_Base_Ensemble_655d.zip
{
 "lambda_min": -62.64004767623212,
 "c": 62.64104767623212,
 "dims": 655,
 "eig_min": 0.0009999999999097012,
 "eig_max": 345.1118164282637,
 "words": 655,
 "dims_file": 655,
 "spearman_cos_vs_score": 0.9999999999970632,
 "cos_min": -0.08368276059627533,
 "cos_max": 0.141214519739151
}

>>> Nên nộp thử file này trước: kỳ vọng ≈ 0.73 (nếu thấp hơn rõ rệt, kiểm tra định dạng 655 chiều).


## 4. Dev set được phép và ví dụ few-shot
Dùng **RG-65** và **MC-30** (tải từ `mfaruqui/eval-word-vectors`; không có WS353/SimLex-999). Dev chỉ dùng để **so sánh các cấu hình Qwen** (không chọn α).
**Few-shot:** chọn `N_SHOTS` cặp RG-65 trải đều theo điểm người chấm (thang 0–4 đổi sang 0–9), **loại** cặp có cả hai từ trong 655 từ; các cặp này bị loại khỏi tập dev đánh giá để công bằng.

In [6]:
RAW = 'https://raw.githubusercontent.com/mfaruqui/eval-word-vectors/master/data/word-sim'
def read_dev_file(path): return [(l.split()[0].lower(), l.split()[1].lower(), float(l.split()[2])) for l in open(path, encoding='utf8') if len(l.split()) >= 3]
DEV = {}
for name, fn in [('RG65', 'EN-RG-65.txt'), ('MC30', 'EN-MC-30.txt')]:
    dest = f'{WORK}/dev_{fn}'
    try:
        if not os.path.exists(dest): urllib.request.urlretrieve(f'{RAW}/{fn}', dest)
        DEV[name] = read_dev_file(dest)
    except Exception as e: print(f'⚠ không tải được {name}: {e}')
assert DEV, 'Không có dev set (cần Internet On)'
assert not any('353' in k or 'simlex' in k.lower() for k in DEV)
DEV_PAIRS = [(a, b, s) for (a, b), s in {(a, b): s for ps in DEV.values() for a, b, s in ps}.items()]

# few-shot: cặp RG-65 trải đều theo điểm, không có cặp nào có cả hai từ trong 655 từ
rg_cand = sorted([(a, b, s) for a, b, s in DEV['RG65'] if not (a in WSET and b in WSET)], key=lambda x: x[2])
pick = [rg_cand[int(round(i * (len(rg_cand) - 1) / (N_SHOTS - 1)))] for i in range(N_SHOTS)]
SHOTS = [(a, b, int(round(s * 9 / 4))) for a, b, s in pick]
SHOT_SET = {(a, b) for a, b, _ in SHOTS} | {(b, a) for a, b, _ in SHOTS}
EXAMPLES_TEXT = 'Reference ratings given by human annotators (same 0-9 scale):\n' + '\n'.join(f'"{a}" / "{b}" -> {s}' for a, b, s in SHOTS) + '\n\n'
DEV_EVAL = [(a, b, s) for a, b, s in DEV_PAIRS if (a, b) not in SHOT_SET]
DEV_GOLD = np.array([s for *_, s in DEV_EVAL])
print({k: len(v) for k, v in DEV.items()}, '| cặp dev duy nhất:', len(DEV_PAIRS), '| dùng để đánh giá (đã loại cặp few-shot):', len(DEV_EVAL))
print('ví dụ few-shot:'); [print(f'   "{a}" / "{b}" -> {s}') for a, b, s in SHOTS]

{'RG65': 65, 'MC30': 30} | cặp dev duy nhất: 69 | dùng để đánh giá (đã loại cặp few-shot): 61
ví dụ few-shot:
   "chord" / "smile" -> 0
   "asylum" / "monk" -> 1
   "grin" / "lad" -> 2
   "cemetery" / "woodland" -> 3
   "magician" / "oracle" -> 4
   "furnace" / "stove" -> 7
   "autograph" / "signature" -> 8
   "midday" / "noon" -> 9


[None, None, None, None, None, None, None, None]

## 5. Ba prompt
Mỗi prompt yêu cầu điểm 0–9 và nhấn mạnh điểm **thấp** cho cặp chỉ liên quan chủ đề hoặc trái nghĩa:
`sim` (giống nghĩa), `subst` (có thay thế được cho nhau trong câu không), `kind` (cùng loại/cùng phạm trù). Điểm = kỳ vọng trên xác suất 10 chữ số, `Σ p_d·d / 9 ∈ [0,1]`.
Các từ ví dụ trong prompt **không thuộc** 655 từ hay dev set (có kiểm tra).

In [7]:
PROMPT_VERSION = 'v2'
SYSTEM_PROMPT = 'You are a careful lexicographer. You judge how similar the meanings of two English words are.'
PROMPTS = {
 'sim': '''{examples}Rate how similar in MEANING the two words are, on a scale from 0 to 9.

9 = synonyms or almost identical meaning (e.g. "couch" / "settee")
6-8 = very close in meaning, the same kind of thing
3-5 = same broad category but clearly different things
1-2 = only loosely related
0 = unrelated

Give a LOW score to words that are merely associated or used together (e.g. "tennis" / "racket") and to opposites (e.g. "ascend" / "descend"), even though they are strongly related.

Word 1: {a}
Word 2: {b}

Answer with a single digit (0-9) and nothing else.''',
 'subst': '''{examples}Rate on a scale from 0 to 9 how well the two words can replace each other in a sentence without changing its meaning.

9 = fully interchangeable synonyms (e.g. "couch" / "settee")
6-8 = interchangeable in most contexts
3-5 = the same general kind of thing, but swapping them changes the meaning
1-2 = rarely substitutable
0 = cannot replace each other at all

Give a LOW score to words that merely occur together or are associated (e.g. "tennis" / "racket") and to opposites (e.g. "ascend" / "descend").

Word 1: {a}
Word 2: {b}

Answer with a single digit (0-9) and nothing else.''',
 'kind': '''{examples}Rate on a scale from 0 to 9 how close the two words are in KIND: do they name the same type of thing, or sibling types of the same category?

9 = the same thing, or near-identical types (e.g. "couch" / "settee")
6-8 = very close sibling types (e.g. "trout" / "salmon")
3-5 = same broad domain but clearly different types (e.g. "trout" / "whale")
1-2 = only loosely related
0 = unrelated

Give a LOW score to words that are merely associated (e.g. "tennis" / "racket") and to opposites (e.g. "ascend" / "descend").

Word 1: {a}
Word 2: {b}

Answer with a single digit (0-9) and nothing else.''',
}
PROMPT_EXAMPLE_WORDS = ['couch', 'settee', 'tennis', 'racket', 'ascend', 'descend', 'trout', 'salmon', 'whale']
DEV_WORDS = {w for a, b, _ in DEV_PAIRS for w in (a, b)}
leak = [w for w in PROMPT_EXAMPLE_WORDS if w in WSET or w in DEV_WORDS]
assert not leak, f'Từ ví dụ trong prompt trùng từ vựng/dev: {leak}'
assert all(p in PROMPTS for p in PROMPT_SET), f'PROMPT_SET chỉ được chọn từ {list(PROMPTS)}'
print('Kiểm tra: từ ví dụ trong prompt không trùng 655 từ hay dev set ✔')
def render_user(name, fewshot, a, b): return PROMPTS[name].format(examples=EXAMPLES_TEXT if fewshot else '', a=a, b=b)

Kiểm tra: từ ví dụ trong prompt không trùng 655 từ hay dev set ✔


## 6. Nạp Qwen (phần cần GPU) và kiểm tra tokenizer

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_NAME); tok.padding_side = 'left'
if tok.pad_token is None: tok.pad_token = tok.eos_token
dt = {'float16': torch.float16, 'bfloat16': torch.bfloat16, 'float32': torch.float32}[DTYPE]
kw = dict(torch_dtype=dt, device_map={'': 0} if SINGLE_GPU else 'auto')
if LOAD_4BIT:
    from transformers import BitsAndBytesConfig
    kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=dt)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kw).eval()

DIGIT_IDS = []
for d in range(10):
    ids = tok.encode(str(d), add_special_tokens=False); assert len(ids) == 1, f'chữ số {d} -> {ids}'; DIGIT_IDS.append(ids[0])
assert len(set(DIGIT_IDS)) == 10; print('token chữ số:', DIGIT_IDS)
_ids_t = torch.tensor(DIGIT_IDS)

def chat_text(system, user):
    return tok.apply_chat_template([{'role': 'system', 'content': system}, {'role': 'user', 'content': user}], tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def score_texts(texts):
    # nhận danh sách văn bản chat đã dựng; trả điểm [0,1] = Σ p_d·d/9 (softmax chỉ trên 10 chữ số)
    enc = tok(texts, return_tensors='pt', padding=True).to(model.device)
    pos = (enc['attention_mask'].cumsum(-1) - 1).clamp(min=0)                       # vị trí đúng khi đệm trái
    base = getattr(model, 'model', None)
    if base is not None and hasattr(model, 'lm_head'):
        h = base(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'], position_ids=pos, use_cache=False).last_hidden_state[:, -1, :]
        Wd = model.lm_head.weight[_ids_t.to(model.lm_head.weight.device)]
        logits = h.to(Wd.device) @ Wd.T
    else:
        logits = model(**enc, use_cache=False).logits[:, -1, :][:, _ids_t.to(model.device)]
    p = torch.softmax(logits.float(), dim=-1)
    return ((p * torch.arange(10, device=p.device)).sum(-1) / 9).cpu().numpy().astype(np.float32)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

token chữ số: [15, 16, 17, 18, 19, 20, 21, 22, 23, 24]


In [9]:
def score_words(pairs, name, fewshot):
    return score_texts([chat_text(SYSTEM_PROMPT, render_user(name, fewshot, a, b)) for a, b in pairs])

print('--- prompt mẫu (sim, zero-shot) ---'); print(render_user('sim', False, 'apple', 'orange')); print('-----------------------------------')
demo = [('couch', 'settee'), ('tennis', 'racket'), ('ascend', 'descend'), ('dog', 'puppy'), ('bank', 'river'), ('happy', 'joyful'), ('tree', 'sandwich')]
for (a, b), s in zip(demo, score_words(demo, 'sim', False)): print(f'  {a:8s} - {b:9s}  điểm Qwen = {s:.3f}  (≈ {9*s:.1f}/9)')
try:
    import torch; print('VRAM cực đại: %.1f GB' % (torch.cuda.max_memory_allocated() / 2**30)) if torch.cuda.is_available() else print('(không có GPU)')
except Exception: pass

--- prompt mẫu (sim, zero-shot) ---
Rate how similar in MEANING the two words are, on a scale from 0 to 9.

9 = synonyms or almost identical meaning (e.g. "couch" / "settee")
6-8 = very close in meaning, the same kind of thing
3-5 = same broad category but clearly different things
1-2 = only loosely related
0 = unrelated

Give a LOW score to words that are merely associated or used together (e.g. "tennis" / "racket") and to opposites (e.g. "ascend" / "descend"), even though they are strongly related.

Word 1: apple
Word 2: orange

Answer with a single digit (0-9) and nothing else.
-----------------------------------
  couch    - settee     điểm Qwen = 1.000  (≈ 9.0/9)
  tennis   - racket     điểm Qwen = 0.181  (≈ 1.6/9)
  ascend   - descend    điểm Qwen = 0.193  (≈ 1.7/9)
  dog      - puppy      điểm Qwen = 0.952  (≈ 8.6/9)
  bank     - river      điểm Qwen = 0.332  (≈ 3.0/9)
  happy    - joyful     điểm Qwen = 1.000  (≈ 9.0/9)
  tree     - sandwich   điểm Qwen = 0.110  (≈ 1.0/9)
VRAM 

## 7. Cache theo thành phần và chạy theo batch có checkpoint
Khoá cache = băm của **(mô hình, 4-bit, kiểu số, phiên bản prompt, system prompt, văn bản prompt, few-shot có/không và nội dung ví dụ, danh sách cặp theo đúng thứ tự)**. Thứ tự `ba` dùng danh sách cặp đảo nên có khoá riêng.
Điểm lưu ở `qwen_<loại>_<khoá>.npy` (NaN = chưa chấm), ghi nguyên tử. Chạy lại chỉ chấm cặp còn NaN; `STOP_AFTER_HOURS` dừng êm và lưu.

In [10]:
from tqdm.auto import tqdm
def cache_key(kind, name, fewshot, pw):
    h = hashlib.sha1()
    for s in [MODEL_NAME, str(LOAD_4BIT), DTYPE, PROMPT_VERSION, SYSTEM_PROMPT, PROMPTS[name], EXAMPLES_TEXT if fewshot else '-', kind, '|'.join(f'{a},{b}' for a, b in pw)]:
        h.update(s.encode('utf8')); h.update(b'\x00')
    return h.hexdigest()[:16]

def load_cache(kind, name, fewshot, pw):
    key = cache_key(kind, name, fewshot, pw); f = f'{CACHE_DIR}/qwen_{kind}_{key}.npy'
    if not os.path.exists(f) and INPUT_CACHE_DIR:
        src = f'{INPUT_CACHE_DIR}/qwen_{kind}_{key}.npy'
        if os.path.exists(src): shutil.copy(src, f); print(f'  đã chép cache từ {src}')
    if os.path.exists(f):
        arr = np.load(f); assert arr.shape == (len(pw),), 'cache sai kích thước'
    else: arr = np.full(len(pw), np.nan, np.float32)
    return key, f, arr

def save_cache(f, arr, key, kind, name, fewshot):
    np.save(f + '.tmp.npy', arr); os.replace(f + '.tmp.npy', f)
    json.dump(dict(model=MODEL_NAME, key=key, kind=kind, prompt=name, fewshot=fewshot, n=len(arr), done=int((~np.isnan(arr)).sum()), version=PROMPT_VERSION, load_4bit=LOAD_4BIT, dtype=DTYPE), open(f.replace('.npy', '.json'), 'w'))

def run_scoring(kind, pw, name, fewshot, sample=None, deadline=None, quiet=False):
    key, f, arr = load_cache(kind, name, fewshot, pw); todo = np.flatnonzero(np.isnan(arr))
    if not quiet: print(f'[{kind}/{name}{"+fs" if fewshot else ""}] đã có {len(arr)-len(todo):,}/{len(arr):,} cặp (khoá {key})')
    if sample is not None and len(todo) > 0: todo = np.sort(np.random.default_rng(0).choice(todo, size=min(sample, len(todo)), replace=False))
    t0 = time.time(); last = 0; done = 0; stopped = False
    for s in tqdm(range(0, len(todo), BATCH_SIZE), desc=f'Qwen[{kind}/{name}]', disable=quiet):
        if deadline is not None and time.time() > deadline: stopped = True; break
        idx = todo[s:s + BATCH_SIZE]; sc = score_words([pw[i] for i in idx], name, fewshot)
        assert np.isfinite(sc).all(), 'Gặp NaN/inf: thử DTYPE="float32"/"bfloat16" hoặc giảm BATCH_SIZE'
        arr[idx] = sc; done += len(idx)
        if done - last >= CHECKPOINT_EVERY: save_cache(f, arr, key, kind, name, fewshot); last = done
    if len(todo): save_cache(f, arr, key, kind, name, fewshot)
    return arr, (done / max(time.time() - t0, 1e-9) if done else float('nan')), stopped

## 8. So sánh các cấu hình Qwen trên dev (rẻ, chạy luôn)
Chấm các cặp dev bằng **mọi tổ hợp** prompt × (zero-shot / few-shot) × thứ tự, rồi in Spearman của từng cấu hình riêng và của trung bình. Dev nhỏ (≈ 60 cặp, sai số ≈ ±0.03): **chỉ dùng để loại cấu hình tệ hoặc lỗi**, không phải để chọn tinh.
Bảng giúp bạn quyết định `PROMPT_SET`, `BOTH_ORDERS`, `USE_FEWSHOT` trước khi tốn giờ GPU.

In [11]:
DEV_AB = [(a, b) for a, b, _ in DEV_EVAL]; DEV_BA = [(b, a) for a, b in DEV_AB]
rho = lambda s: float(spearmanr(s, DEV_GOLD)[0])
dev_arr = {}
for name in PROMPTS:
    for fs in (False, True):
        for od, pw in (('ab', DEV_AB), ('ba', DEV_BA)):
            dev_arr[(name, fs, od)] = run_scoring('dev', pw, name, fs, quiet=True)[0]
print(f'{"prompt":8s} {"few-shot":9s} | {"(A,B)":>6s} {"(B,A)":>6s} {"TB hai thứ tự":>14s}   [Spearman trên {len(DEV_EVAL)} cặp dev]')
for fs in (False, True):
    for name in PROMPTS:
        ab, ba = dev_arr[(name, fs, 'ab')], dev_arr[(name, fs, 'ba')]
        print(f'{name:8s} {"có" if fs else "không":9s} | {rho(ab):6.3f} {rho(ba):6.3f} {rho((ab+ba)/2):14.3f}')
    allm = np.mean([(dev_arr[(nm, fs, "ab")] + dev_arr[(nm, fs, "ba")]) / 2 for nm in PROMPTS], axis=0)
    print(f'{"cả 3":8s} {"có" if fs else "không":9s} | {"":6s} {"":6s} {rho(allm):14.3f}   <- trung bình cả 3 prompt, cả 2 thứ tự')
cfg_comps = [dev_arr[(p, USE_FEWSHOT, o)] for p in PROMPT_SET for o in (['ab', 'ba'] if BOTH_ORDERS else ['ab'])]
q_dev = rho(np.mean(cfg_comps, axis=0))
print(f'\nCấu hình hiện tại (PROMPT_SET={PROMPT_SET}, BOTH_ORDERS={BOTH_ORDERS}, USE_FEWSHOT={USE_FEWSHOT}): Spearman Qwen trên dev = {q_dev:.3f}')
if q_dev < ABORT_IF_QWEN_DEV_BELOW:
    raise RuntimeError(f'Qwen chỉ đạt {q_dev:.3f} trên dev (< {ABORT_IF_QWEN_DEV_BELOW}): kiểm tra prompt/tokenizer/dtype trước khi chấm 214.185 cặp.')

prompt   few-shot  |  (A,B)  (B,A)  TB hai thứ tự   [Spearman trên 61 cặp dev]
sim      không     |  0.906  0.893          0.909
subst    không     |  0.894  0.873          0.905
kind     không     |  0.908  0.913          0.921
cả 3     không     |                        0.914   <- trung bình cả 3 prompt, cả 2 thứ tự
sim      có        |  0.734  0.738          0.772
subst    có        |  0.758  0.744          0.791
kind     có        |  0.746  0.824          0.801
cả 3     có        |                        0.784   <- trung bình cả 3 prompt, cả 2 thứ tự

Cấu hình hiện tại (PROMPT_SET=['sim'], BOTH_ORDERS=True, USE_FEWSHOT=False): Spearman Qwen trên dev = 0.909


## 9. Kế hoạch thành phần và ước tính giờ
`COMPONENTS` = các cặp (prompt, thứ tự) theo cấu hình, chạy lần lượt (các thứ tự của cùng prompt đứng cạnh nhau, nên dù chỉ xong một phần thì vẫn có điểm trung bình hai thứ tự cho prompt đầu).
- `SMOKE_TEST = True`: chấm `N_SMOKE` cặp ngẫu nhiên của thành phần đầu, đo tốc độ và in ước tính giờ cho từng thành phần.
- `SMOKE_TEST = False`: chạy đủ các thành phần (tối đa `COMPONENTS_PER_RUN`), dừng êm khi hết `STOP_AFTER_HOURS`.

In [12]:
def vram_gb():
    try:
        import torch
        return f'{torch.cuda.max_memory_allocated()/2**30:.1f} GB' if torch.cuda.is_available() else 'n/a (không có GPU)'
    except Exception: return 'n/a'
COMPONENTS = [(p, o) for p in PROMPT_SET for o in (['ab', 'ba'] if BOTH_ORDERS else ['ab'])]
PW = {'ab': PAIR_WORDS, 'ba': PAIR_WORDS_BA}
def comp_cache(p, o): key, f, arr = load_cache('main', p, USE_FEWSHOT, PW[o]); return arr
print(f'Thành phần theo cấu hình ({len(COMPONENTS)}): {COMPONENTS}')
print('Trạng thái cache hiện có:'); [print(f'  {p}/{o}: {int((~np.isnan(comp_cache(p, o))).sum()):,}/{NP:,} cặp') for p, o in COMPONENTS]

if SMOKE_TEST:
    p0, o0 = COMPONENTS[0]
    _, rate, _ = run_scoring('main', PW[o0], p0, USE_FEWSHOT, sample=N_SMOKE)
    if rate == rate:
        print(f'\n[SMOKE] tốc độ ≈ {rate:.1f} cặp/giây (prompt {p0}, few-shot={USE_FEWSHOT}); VRAM cực đại: {vram_gb()}')
        per = NP / rate / 3600; print(f'{"thành phần":14s} {"giờ (ước tính)":>15s} {"cộng dồn":>9s}')
        cum = 0
        for p, o in COMPONENTS:
            cum += per; print(f'{p+"/"+o:14s} {per:15.1f} {cum:9.1f}' + ('   ⚠ vượt SESSION_HOURS_BUDGET: chia nhiều phiên bằng COMPONENTS_PER_RUN' if cum > SESSION_HOURS_BUDGET else ''))
        print('(Ước tính thô từ một mẫu ngẫu nhiên; prompt few-shot dài hơn nên chậm hơn. Giảm PROMPT_SET/BOTH_ORDERS hoặc dùng mô hình 3B nếu quá lâu.)')

Thành phần theo cấu hình (2): [('sim', 'ab'), ('sim', 'ba')]
Trạng thái cache hiện có:
  sim/ab: 0/214,185 cặp
  sim/ba: 0/214,185 cặp


In [13]:
STOPPED = False
if not SMOKE_TEST:
    deadline = None if STOP_AFTER_HOURS is None else T_START + STOP_AFTER_HOURS * 3600
    todo_comps = [(p, o) for p, o in COMPONENTS if np.isnan(comp_cache(p, o)).any()]
    if COMPONENTS_PER_RUN is not None: todo_comps = todo_comps[:COMPONENTS_PER_RUN]
    print(f'Sẽ chạy {len(todo_comps)} thành phần chưa hoàn tất: {todo_comps}')
    for p, o in todo_comps:
        arr, rate, STOPPED = run_scoring('main', PW[o], p, USE_FEWSHOT, deadline=deadline)
        print(f'  {p}/{o}: ' + ('đủ' if not np.isnan(arr).any() else f'CHƯA đủ ({int((~np.isnan(arr)).sum()):,}/{NP:,})') + (f', ≈ {rate:.1f} cặp/giây' if rate == rate else ''))
        if STOPPED: print('⚠ Đã chạm STOP_AFTER_HOURS: dừng êm, cache đã lưu. Chạy lại (hoặc phiên sau) để tiếp tục.'); break
COMPLETE = [(p, o) for p, o in COMPONENTS if not np.isnan(comp_cache(p, o)).any()]
print('\nThành phần đã chấm ĐỦ 214.185 cặp:', COMPLETE or 'chưa có')

Sẽ chạy 2 thành phần chưa hoàn tất: [('sim', 'ab'), ('sim', 'ba')]
[main/sim] đã có 0/214,185 cặp (khoá 3ec1f0d05183df82)


Qwen[main/sim]:   0%|          | 0/6694 [00:00<?, ?it/s]

  sim/ab: đủ, ≈ 6.0 cặp/giây
[main/sim] đã có 0/214,185 cặp (khoá 387a9a5898f618d3)


Qwen[main/sim]:   0%|          | 0/6694 [00:00<?, ?it/s]

  sim/ba: CHƯA đủ (10,208/214,185), ≈ 6.0 cặp/giây
⚠ Đã chạm STOP_AFTER_HOURS: dừng êm, cache đã lưu. Chạy lại (hoặc phiên sau) để tiếp tục.

Thành phần đã chấm ĐỦ 214.185 cặp: [('sim', 'ab')]


## 10. Ghép và xuất file nộp
Chỉ dùng các thành phần **đã chấm đủ**. Mỗi thành phần được z-score trên 214.185 cặp, lấy **trung bình** rồi z-score lại thành `z(S_qwen)`.
- `WE_Qwen_only.zip`: chỉ `z(S_qwen)` — để đo sức mạnh riêng của Qwen.
- `WE_Combined_alpha<α>.zip`: `z(S_ens) + α·z(S_qwen)` cho mỗi α trong `ALPHA_LIST`.
Không xuất nếu đang `SMOKE_TEST` hoặc chưa có thành phần nào đủ.

In [14]:
STATUS.update({'smoke_test': SMOKE_TEST, 'components_complete': COMPLETE, 'components_planned': COMPONENTS, 'time_stopped': STOPPED})
qwen_zips = []
if SMOKE_TEST: print('SMOKE_TEST = True: KHÔNG xuất bản Qwen.')
elif not COMPLETE: print('Chưa có thành phần nào chấm đủ: KHÔNG xuất bản Qwen (chạy tiếp rồi chạy lại cell này).')
else:
    comps = [zscore(comp_cache(p, o).astype(np.float64))[0] for p, o in COMPLETE]
    ZQ, ZQ_P = zscore(np.mean(comps, axis=0))
    print(f'Dùng {len(COMPLETE)} thành phần: {COMPLETE}' + ('' if len(COMPLETE) == len(COMPONENTS) else f'  (⚠ thiếu {len(COMPONENTS)-len(COMPLETE)} thành phần chưa đủ)'))
    if len(COMPLETE) > 1: print('tương quan giữa các thành phần (Spearman):', np.round(spearmanr(np.array(comps).T)[0], 3).tolist() if len(COMPLETE) > 2 else round(float(spearmanr(comps[0], comps[1])[0]), 3))
    print(f'tương quan Spearman giữa Qwen và ensemble trên 214.185 cặp: {spearmanr(ZQ, ZE)[0]:.3f}  (thấp = Qwen cho thông tin khác ensemble; rất cao = ít bổ sung)')
    def exp(score, path):
        info = export_zip(score, path); print(f'  đã xuất {path}: chiều {info["dims_file"]}, Spearman(cos, điểm) = {info["spearman_cos_vs_score"]:.6f}'); return path
    qwen_zips.append(exp(ZQ, f'{OUT_DIR}/WE_Qwen_only.zip'))
    for a in ALPHA_LIST:
        if a > 0: qwen_zips.append(exp(ZE + a * ZQ, f'{OUT_DIR}/WE_Combined_alpha{a:g}.zip'))
STATUS['qwen_zips'] = qwen_zips

Dùng 1 thành phần: [('sim', 'ab')]  (⚠ thiếu 1 thành phần chưa đủ)


tương quan Spearman giữa Qwen và ensemble trên 214.185 cặp: 0.309  (thấp = Qwen cho thông tin khác ensemble; rất cao = ít bổ sung)


  đã xuất /kaggle/working/out/WE_Qwen_only.zip: chiều 655, Spearman(cos, điểm) = 1.000000


  đã xuất /kaggle/working/out/WE_Combined_alpha0.5.zip: chiều 655, Spearman(cos, điểm) = 1.000000


  đã xuất /kaggle/working/out/WE_Combined_alpha1.zip: chiều 655, Spearman(cos, điểm) = 1.000000
  đã xuất /kaggle/working/out/WE_Combined_alpha2.zip: chiều 655, Spearman(cos, điểm) = 1.000000


## 11. Tổng kết — cái đã chạy trong phiên này và cái bạn sẽ nhận

In [15]:
print('=' * 72); print('CẤU HÌNH')
print(f'  mô hình={MODEL_NAME} | 4bit={LOAD_4BIT} | dtype={DTYPE} | batch={BATCH_SIZE} | few-shot={USE_FEWSHOT}')
print(f'  PROMPT_SET={PROMPT_SET} | BOTH_ORDERS={BOTH_ORDERS} | ALPHA_LIST={ALPHA_LIST} | OUT_DIM={OUT_DIM or "đủ 655"} | SMOKE_TEST={SMOKE_TEST}')
print('TRẠNG THÁI PHIÊN CHẠY NÀY')
print(f'  bản nền 655 chiều đã xuất   : {STATUS.get("base_exported")}  -> {STATUS.get("base_zip")}')
print(f'  thành phần Qwen đã chấm đủ  : {len(STATUS["components_complete"])}/{len(STATUS["components_planned"])}  {STATUS["components_complete"]}')
print(f'  dừng vì hết STOP_AFTER_HOURS: {STATUS["time_stopped"]}')
print(f'  file Qwen đã xuất           : {STATUS["qwen_zips"] or "KHÔNG (smoke-test, hoặc chưa có thành phần nào đủ)"}')
print('FILE TRONG', OUT_DIR); [print('  ', f) for f in sorted(os.listdir(OUT_DIR)) if f.endswith('.zip')]
print('=' * 72)
print('Gợi ý nộp: (1) WE_Base_Ensemble_655d (≈0.73, kiểm tra định dạng) -> (2) WE_Qwen_only -> (3) WE_Combined_alpha*.')
print('Nhắc lại: điểm dev chỉ là kiểm tra cục bộ; điểm thật chỉ biết sau khi nộp.')

CẤU HÌNH
  mô hình=Qwen/Qwen2.5-7B-Instruct | 4bit=True | dtype=float16 | batch=32 | few-shot=False
  PROMPT_SET=['sim'] | BOTH_ORDERS=True | ALPHA_LIST=[0.5, 1, 2] | OUT_DIM=đủ 655 | SMOKE_TEST=False
TRẠNG THÁI PHIÊN CHẠY NÀY
  bản nền 655 chiều đã xuất   : True  -> /kaggle/working/out/WE_Base_Ensemble_655d.zip
  thành phần Qwen đã chấm đủ  : 1/2  [('sim', 'ab')]
  dừng vì hết STOP_AFTER_HOURS: True
  file Qwen đã xuất           : ['/kaggle/working/out/WE_Qwen_only.zip', '/kaggle/working/out/WE_Combined_alpha0.5.zip', '/kaggle/working/out/WE_Combined_alpha1.zip', '/kaggle/working/out/WE_Combined_alpha2.zip']
FILE TRONG /kaggle/working/out
   WE_Base_Ensemble_655d.zip
   WE_Combined_alpha0.5.zip
   WE_Combined_alpha1.zip
   WE_Combined_alpha2.zip
   WE_Qwen_only.zip
Gợi ý nộp: (1) WE_Base_Ensemble_655d (≈0.73, kiểm tra định dạng) -> (2) WE_Qwen_only -> (3) WE_Combined_alpha*.
Nhắc lại: điểm dev chỉ là kiểm tra cục bộ; điểm thật chỉ biết sau khi nộp.
